In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider


# Set Up the Engine and Loads the NLP Model
# Presidio defaultly uses "en_core_web_sm" model Which is weak in PII dectection, "en_core_web_lg" model is better as compared.
provider = NlpEngineProvider(nlp_configuration={
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "en", "model_name": "en_core_web_lg"}]})

nlp_engine = provider.create_engine()
analyzer = AnalyzerEngine(nlp_engine=nlp_engine)

In [56]:
text = "My name is Mahesh and I paid ₹500 to zomato and it got failed"

# Call the analyzer to get the result
results = analyzer.analyze(
    text = text,
    # Only few supported Entities
    entities = ["PERSON", "PHONE_NUMBER", "AMOUNT", "LOCATION"],
    language = "en"
)
print(results)

[type: PERSON, start: 11, end: 17, score: 0.85]


In [57]:
# Analyzer results are passed to AnonymizerEngine for Anonymization
anonymizer = AnonymizerEngine()
anonymized_text = anonymizer.anonymize(text=text, analyzer_results=results)

print(anonymized_text)

text: My name is <PERSON> and I paid ₹500 to zomato and it got failed
items:
[
    {'start': 11, 'end': 19, 'entity_type': 'PERSON', 'text': '<PERSON>', 'operator': 'replace'}
]



## Custom Entities

In [58]:
from presidio_analyzer import Pattern, PatternRecognizer

# 1. UPI ID 
upi_pattern = Pattern(
    name="UPI_Pattern",
    regex=r"[a-zA-Z0-9.\-_]{2,256}@[a-zA-Z]{2,64}",
    score=0.85
)
upi_recognizer = PatternRecognizer(supported_entity="UPI_ID", patterns=[upi_pattern])

# 2. Merchant Name 
merchant_pattern = Pattern(
    name="Merchant_Pattern",
    regex=r"\b(Zomato|Swiggy|PhonePe|Paytm|Amazon|Flipkart|IRCTC)\b",
    score=0.75
)
merchant_recognizer = PatternRecognizer(supported_entity="MERCHANT_NAME", patterns=[merchant_pattern])

# 3. Person Name 
name_pattern = Pattern(
    name="Name_Pattern",
    regex=r"\b(Mr\.|Mrs\.|Ms\.|Dr\.)\s[A-Z][a-z]+(?:\s[A-Z][a-z]+)?\b",
    score=0.70
)
name_recognizer = PatternRecognizer(supported_entity="PERSON", patterns=[name_pattern])


# 4. IFSC Code 
ifsc_pattern = Pattern(
    name="IFSC_Pattern",
    regex=r"\b[A-Z]{4}0[A-Z0-9]{6}\b",
    score=0.90
)
ifsc_recognizer = PatternRecognizer(supported_entity="IFSC_CODE", patterns=[ifsc_pattern])

# 5. CVV Recognizer 
cvv_pattern = Pattern(
    name="CVV_Pattern",
    regex=r"\bcvv[:\s]?\d{3,4}\b",  
    score=0.85
)
cvv_recognizer = PatternRecognizer(supported_entity="CVV", patterns=[cvv_pattern], context=["cvv", "security code"])

# 6. OTP Recognizer 
otp_pattern = Pattern(
    name="OTP_Pattern",
    regex=r"\b\d{4,6}\b",
    score=0.55
)
otp_recognizer = PatternRecognizer(supported_entity="OTP", patterns=[otp_pattern], context=["otp", "one time password", "verification code"])

# 7. PIN Recognizer 
pin_pattern = Pattern(
    name="PIN_Pattern",
    regex=r"\b\d{4,6}\b",
    score=0.55
)
pin_recognizer = PatternRecognizer(supported_entity="PIN", patterns=[pin_pattern], context=["pin", "mpin", "atm pin"])

# 8. Transaction ID 
txn_pattern = Pattern(
    name="TXN_Pattern",
    regex=r"\b(TXN|UTR|REF)[A-Z0-9]{8,20}\b",
    score=0.88
)
txn_recognizer = PatternRecognizer(supported_entity="TRANSACTION_ID",patterns=[txn_pattern])

# 9. Amount Recognizer ─────────────────────────────────────────────────────
amount_pattern = Pattern(
    name="Amount_Pattern",
    regex=r"(?:Rs\.?|INR|₹)\s?\d+(?:,\d{3})*(?:\.\d{1,2})?|\b\d+(?:,\d{3})*(?:\.\d{1,2})?\s?(?:rupees|rs)",
    score=0.80
)
amount_recognizer = PatternRecognizer(supported_entity="AMOUNT",patterns=[amount_pattern],
    context=["payment", "paid", "charged", "debited", "credited", "transferred", "amount", "cost", "price"])


for recognizer in [
    upi_recognizer,
    merchant_recognizer,
    name_recognizer,
    ifsc_recognizer,
    cvv_recognizer,
    otp_recognizer,
    pin_recognizer,
    txn_recognizer,
    amount_recognizer,
]:
    analyzer.registry.add_recognizer(recognizer)


In [59]:
# Test 
text = (
    "Mr. Mahesh payment of ₹500 to Zomato via mahesh@okaxis failed. "
)

ALL_ENTITIES = [
    "UPI_ID", "MERCHANT_NAME", "PERSON",
    "IFSC_CODE", "CVV", "OTP", "PIN", "TRANSACTION_ID", "AMOUNT"
]

results = analyzer.analyze(text=text, entities=ALL_ENTITIES, language="en")
anon_result = anonymizer.anonymize(text=text, analyzer_results=results)
anon_text = anon_result.text
anon_text

'<PERSON> of <AMOUNT> to <MERCHANT_NAME> via <UPI_ID> failed. '

In [60]:
from collections import defaultdict

# Palceholder Mapping
placeholder_map = defaultdict(list)
for item in results: 
    placeholder = f"<{item.entity_type}>"
    original = text[item.start:item.end]   
    placeholder_map[placeholder].append(original)

placeholder_map

defaultdict(list,
            {'<AMOUNT>': ['₹500'],
             '<PERSON>': ['Mahesh', 'Mr. Mahesh payment'],
             '<UPI_ID>': ['mahesh@okaxis'],
             '<MERCHANT_NAME>': ['Zomato']})

## Passing to LLM

In [61]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

LLM = ChatGroq(
    model = "llama-3.3-70b-versatile",
    api_key = os.getenv("GROQ_API_KEY"),
    temperature = 0.4)

system_prompt = """You are a secure, privacy-aware customer support assistant for a digital payments platform.
User messages are pre-processed by a PII anonymization layer before reaching you.
Sensitive values are replaced with typed placeholders in this exact format:
Like <PERSON>, <AMOUNT>, <MERCHANT_NAME>, <BANK_ACCOUNT>, <OTP>, <PIN> and more..         
Your behavior rules:
1. Treat every placeholder as a real value — never question, reveal, or comment on them.
2. Use placeholders naturally in your response exactly as they appear (e.g. "your payment of <AMOUNT> to <MERCHANT_NAME>").
3. Never generate, guess, or fabricate values to fill placeholders.
4. If a placeholder appears in your response, reproduce it exactly — same case, same format.
5. Stay focused on resolving the payment issue — do not ask for sensitive data.
6. Keep responses concise (2-4 sentences max) and empathetic in tone.
Tone: professional, calm, helpful."""

template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{user_input}")])

chain = template | LLM
response = chain.invoke({"user_input": anon_text})
print("LLM Response with Placeholders:")
result = response.content
print(result)


LLM Response with Placeholders:
I'm sorry to hear that your payment of <AMOUNT> to <MERCHANT_NAME> via <UPI_ID> was unsuccessful. I'm here to help you resolve the issue. Can you please tell me more about the error message you received or what happened when the payment failed? I'll do my best to assist you in completing the transaction.


In [62]:
# Replacing the PlaceHolders
rehydrated = result
for placeholder, originals in placeholder_map.items():
    for original in originals:
        rehydrated = rehydrated.replace(placeholder, original, 1)

rehydrated

"I'm sorry to hear that your payment of ₹500 to Zomato via mahesh@okaxis was unsuccessful. I'm here to help you resolve the issue. Can you please tell me more about the error message you received or what happened when the payment failed? I'll do my best to assist you in completing the transaction."